In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import os
from nn_core.common import PROJECT_ROOT
from pytorch_lightning import seed_everything
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import AutoConfig, AutoModel, AutoImageProcessor
import numpy as np
import functools
import matplotlib.pyplot as plt
import seaborn as sns
from layskip.utils.utils import image_encode, extract_all_layers
from layskip.utils.dictionaries import (
    DATASET2INPUT_COLUMN,
    DATASET2LABEL_COLUMN,
    MODEL2NUM_LAYERS,
    MODEL2CONFIGS,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
def linear_approximation_error(X, Y):
    """
    Measure how well a linear transformation can map X to Y.
    This is exactly what your linear translator minimizes.

    Args:
        X: source layer [n_samples, n_tokens, n_features]
        Y: target layer [n_samples, n_tokens, n_features]

    Returns:
        Normalized residual error after optimal linear fit
    """
    # Flatten to match translator training (mode=1)
    X_flat = X.view(-1, X.size(-1))  # [n_samples * n_tokens, n_features]
    Y_flat = Y.view(-1, Y.size(-1))

    # Solve least squares: X @ W ≈ Y
    # This is what lstsq_align_state does
    W = torch.linalg.lstsq(X_flat, Y_flat).solution

    # Compute residual: ||Y - X @ W||_F / ||Y||_F
    Y_pred = X_flat @ W
    residual = torch.norm(Y_flat - Y_pred, p="fro") / torch.norm(Y_flat, p="fro")

    return residual.item()


def procrustes_distance(X, Y):
    """
    Compute Procrustes distance - optimal alignment error.
    Measures how well two spaces can be aligned after optimal rotation.
    """
    X_flat = X.view(-1, X.size(-1))
    Y_flat = Y.view(-1, Y.size(-1))

    # Center
    X_flat = X_flat - X_flat.mean(dim=0, keepdim=True)
    Y_flat = Y_flat - Y_flat.mean(dim=0, keepdim=True)

    # Normalize
    X_flat = X_flat / (torch.norm(X_flat, p="fro") + 1e-8)
    Y_flat = Y_flat / (torch.norm(Y_flat, p="fro") + 1e-8)

    # Optimal rotation via SVD
    U, S, Vt = torch.svd(X_flat.T @ Y_flat)

    # Procrustes distance
    distance = torch.sqrt(torch.clamp(1 - S.sum() ** 2, min=0))

    return distance.item()


def linear_cka(X, Y, centered=True):
    """
    Compute Linear CKA (correct formula).
    CKA formula: ||X^T Y||_F^2 / (||X^T X||_F * ||Y^T Y||_F)

    Args:
        centered: If True, uses centered CKA (standard).
                  If False, uncentered CKA (closer to translator behavior - no mean subtraction)
    """
    X_flat = X.view(-1, X.size(-1))
    Y_flat = Y.view(-1, Y.size(-1))

    # Center the representations (optional - matches translator when False)
    if centered:
        X_flat = X_flat - X_flat.mean(dim=0, keepdim=True)
        Y_flat = Y_flat - Y_flat.mean(dim=0, keepdim=True)

    # Compute gram matrices
    XTX = X_flat.T @ X_flat
    YTY = Y_flat.T @ Y_flat
    XTY = X_flat.T @ Y_flat

    # CKA formula using Frobenius norms
    numerator = torch.norm(XTY, p="fro") ** 2
    denominator = torch.norm(XTX, p="fro") * torch.norm(YTY, p="fro")

    return (numerator / denominator).item()


def cka_distance(X, Y, centered=True):
    """
    1 - CKA gives a distance metric (lower = more similar)

    Args:
        centered: If True, uses centered CKA. If False, uncentered (matches translator).
    """
    return 1.0 - linear_cka(X, Y, centered=centered)


def uncentered_cka_distance(X, Y):
    """Uncentered CKA distance - matches translator behavior (no mean subtraction)"""
    return cka_distance(X, Y, centered=False)


def global_mse(X, Y):
    """
    Compute global MSE between representations.
    Simple baseline metric.
    """
    X_flat = X.view(-1, X.size(-1))
    Y_flat = Y.view(-1, Y.size(-1))

    mse = ((X_flat - Y_flat) ** 2).mean()
    return mse.item()


def global_cosine_similarity(X, Y):
    """
    Compute global cosine similarity.
    Average cosine similarity across all vectors.
    """
    X_flat = X.view(-1, X.size(-1))
    Y_flat = Y.view(-1, Y.size(-1))

    # Normalize each vector
    X_norm = X_flat / (X_flat.norm(dim=-1, keepdim=True) + 1e-8)
    Y_norm = Y_flat / (Y_flat.norm(dim=-1, keepdim=True) + 1e-8)

    # Compute cosine similarity for each pair and take mean
    cosine = (X_norm * Y_norm).sum(dim=-1).mean()

    return cosine.item()


def cosine_distance(X, Y):
    """1 - cosine similarity gives a distance metric"""
    return 1.0 - global_cosine_similarity(X, Y)

In [ ]:
def compute_all_skip_metrics(layers):
    """
    Compute all skip quality metrics for all layer pairs.
    OPTIMIZED: Only computes upper triangle (i < j) to avoid redundant calculations.

    Args:
        layers: [n_layers, n_samples, n_tokens, n_features] tensor

    Returns:
        Dictionary of metric matrices [n_layers, n_layers]
    """
    n_layers = layers.shape[0]

    metrics = {
        "linear_error": torch.zeros((n_layers, n_layers)),
        "procrustes": torch.zeros((n_layers, n_layers)),
        "cka_distance": torch.zeros((n_layers, n_layers)),
        "uncentered_cka_distance": torch.zeros((n_layers, n_layers)),
        "mse": torch.zeros((n_layers, n_layers)),
        "cosine_distance": torch.zeros((n_layers, n_layers)),
    }

    # Count total pairs for progress reporting
    total_pairs = n_layers * (n_layers - 1) // 2  # Only upper triangle
    computed = 0

    print(f"Computing metrics for {n_layers} layers...")
    print(f"Total layer pairs to evaluate: {total_pairs}")
    print("(Only computing upper triangle: i < j)")

    for i in range(n_layers):
        for j in range(i + 1, n_layers):  # OPTIMIZATION: Only j > i (upper triangle)
            layer_i = layers[i].to(device)
            layer_j = layers[j].to(device)

            # Compute all metrics for this pair
            metrics["linear_error"][i, j] = linear_approximation_error(layer_i, layer_j)
            metrics["procrustes"][i, j] = procrustes_distance(layer_i, layer_j)
            metrics["cka_distance"][i, j] = cka_distance(layer_i, layer_j, centered=True)
            metrics["uncentered_cka_distance"][i, j] = uncentered_cka_distance(layer_i, layer_j)
            metrics["mse"][i, j] = global_mse(layer_i, layer_j)
            metrics["cosine_distance"][i, j] = cosine_distance(layer_i, layer_j)

            computed += 1

            # Progress reporting every 10 pairs
            if computed % 10 == 0 or computed == total_pairs:
                print(f"  Progress: {computed}/{total_pairs} pairs ({100*computed/total_pairs:.1f}%)")

    print("Done!")
    print(
        f"Speedup: Computed {computed} pairs instead of {n_layers * n_layers} (saved {n_layers * n_layers - computed} redundant calculations)"
    )

    return metrics

In [ ]:
seed = 42
seed_everything(seed)

# ============================================================================
# CONFIGURATION: Choose your model and dataset
# ============================================================================
# VISION MODELS:
# - "facebook/deit-small-patch16-224" (12 layers, vision)
# - "facebook/dinov2-small" (12 layers, vision)
# - "WinKawaks/vit-small-patch16-224" (12 layers, vision)

# TEXT MODELS:
# - "answerdotai/ModernBERT-base" (22 layers, text)

# VISION DATASETS: "cifar100", "cifar10"
# TEXT DATASETS: "ag_news", "sst2"

# ============================================================================
# SELECT MODEL AND DATASET TYPE
# ============================================================================
model_type = "vision"  # "vision" or "text"

if model_type == "vision":
    model_name = "facebook/dinov2-base"
    dataset_name = "cifar100"
elif model_type == "text":
    model_name = "answerdotai/ModernBERT-base"
    dataset_name = "ag_news"

max_samples = 50

print(f"Model Type: {model_type}")
print(f"Model: {model_name}")
print(f"Dataset: {dataset_name}")
print(f"Samples: {max_samples}")
print(f"\nNote: Using {max_samples} samples provides a good balance between speed and reliability.")
print(f"      - Use 50 samples for quick exploration (~5-10 min)")
print(f"      - Use 100 samples for reasonable estimates (~10-20 min)")
print(f"      - Use 500+ samples for reliable rankings (~30+ min)")

# Get dataset configuration
text_or_image_name = DATASET2INPUT_COLUMN.get(dataset_name)
label_name = DATASET2LABEL_COLUMN.get(dataset_name)
original_num_layers = MODEL2NUM_LAYERS[model_name]

# Load model and processor/tokenizer
config = AutoConfig.from_pretrained(model_name, output_hidden_states=True, return_dict=True)

if model_type == "text":
    from transformers import AutoTokenizer
    from layskip.utils.text_encode_and_skip import extract_text_representations
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    encoder = AutoModel.from_pretrained(model_name, config=config)
    
    # Load dataset
    dataset = load_dataset(dataset_name)
    dataset_train = dataset["train"]
    
    # Tokenization function
    def tokenize_fn(examples):
        texts = examples[text_or_image_name]
        return tokenizer(
            texts,
            padding="max_length",
            truncation=True,
            max_length=128,
            return_tensors="pt",
        )
    
    # Tokenize dataset
    tokenized_train = dataset_train.map(tokenize_fn, batched=True, remove_columns=[text_or_image_name])
    tokenized_train.set_format("torch")
    
    # Create dataloader
    dataloader = DataLoader(
        tokenized_train,
        batch_size=32,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
    )
else:
    # Vision models
    processor = AutoImageProcessor.from_pretrained(model_name)
    encoder = AutoModel.from_pretrained(model_name, config=config)
    
    # Load dataset
    dataset = load_dataset(dataset_name)
    dataset_train = dataset["train"]
    
    # Create dataloader for images
    dataloader = DataLoader(
        dataset_train,
        batch_size=256,
        shuffle=True,
        num_workers=8,
        pin_memory=True,
        collate_fn=functools.partial(image_encode, processor=processor, image_name=text_or_image_name, label_name=label_name),
    )

encoder.to(device)
encoder.eval()

# Extract layer embeddings
print("\nExtracting layer representations...")

if model_type == "text":
    # Use the production-ready extract_text_representations function
    # Note: This already skips embeddings and returns only transformer layer outputs
    layer_embeddings = extract_text_representations(
        model=encoder,
        config=config,
        dataloader=dataloader,
        num_samples=max_samples
    )
else:
    # For vision models, use extract_all_layers (also skips embeddings)
    layer_embeddings = extract_all_layers(
        model=encoder,
        max_samples=max_samples,
        dataloader=dataloader,
        only_cls=False,
    )

print(f"\nExtracted {len(layer_embeddings)} layers")
assert len(layer_embeddings) == original_num_layers, f"Expected {original_num_layers} layers, got {len(layer_embeddings)}"

layer_embeddings = [layer_output for _, layer_output in sorted(layer_embeddings.items())]
layer_embeddings_stacked = torch.stack(layer_embeddings)

print(f"\nLayer embeddings shape: {layer_embeddings_stacked.shape}")
print(
    f"  [n_layers={layer_embeddings_stacked.shape[0]}, n_samples={layer_embeddings_stacked.shape[1]}, n_tokens={layer_embeddings_stacked.shape[2]}, n_features={layer_embeddings_stacked.shape[3]}]"
)
print(f"\nTotal data points per layer pair: {layer_embeddings_stacked.shape[1] * layer_embeddings_stacked.shape[2]:,}")

In [ ]:
# Compute all metrics
skip_metrics = compute_all_skip_metrics(layer_embeddings_stacked)

# Convert to numpy for plotting
skip_metrics_np = {k: v.cpu().numpy() for k, v in skip_metrics.items()}

print("\nMetric ranges:")
for name, matrix in skip_metrics_np.items():
    print(f"  {name:20s}: [{matrix.min():.4f}, {matrix.max():.4f}]")

In [ ]:
# Plot all metrics
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

metric_configs = [
    ("linear_error", "Linear Approximation Error", "viridis_r", "Lower = Better"),
    ("procrustes", "Procrustes Distance", "viridis_r", "Lower = Better"),
    ("cka_distance", "1 - CKA", "viridis_r", "Lower = Better"),
    ("mse", "Mean Squared Error", "viridis_r", "Lower = Better"),
    ("cosine_distance", "1 - Cosine Similarity", "viridis_r", "Lower = Better"),
]

for idx, (metric_key, title, cmap, interpretation) in enumerate(metric_configs):
    ax = axes[idx]
    matrix = skip_metrics_np[metric_key]

    sns.heatmap(matrix, annot=True, fmt=".3f", cmap=cmap, ax=ax, cbar_kws={"label": metric_key}, annot_kws={"size": 7})

    ax.set_title(f"{title}\n({interpretation})", fontsize=12, fontweight="bold")
    ax.set_xlabel("Target Layer", fontsize=10)
    ax.set_ylabel("Source Layer", fontsize=10)

# Hide the last subplot (we only have 5 metrics)
axes[-1].axis("off")

plt.suptitle(
    f"Skip Quality Metrics Comparison\n{model_name} on {dataset_name} ({max_samples} samples)",
    fontsize=14,
    fontweight="bold",
    y=0.995,
)
plt.tight_layout()
plt.show()

# Save figure
PLOTS_DIR = PROJECT_ROOT / "plots" / "skip_quality_metrics"
os.makedirs(PLOTS_DIR, exist_ok=True)
save_path = PLOTS_DIR / f"{dataset_name}_{model_name.split('/')[1]}_skip_metrics_comparison.png"
fig.savefig(save_path, dpi=300, bbox_inches="tight")
print(f"\nSaved to: {save_path}")

## Identify Best Skip Candidates

For each metric, find the best skip candidates (adjacent layers with lowest distance).

In [ ]:
def find_best_skips(metric_matrix, n_layers, top_k=5, min_gap=1):
    """
    Find best skip candidates based on a metric.

    Args:
        metric_matrix: [n_layers, n_layers] distance matrix (lower = better)
        n_layers: number of layers
        top_k: number of top candidates to return
        min_gap: minimum number of layers between source and target

    Returns:
        List of (source, target, score) tuples
    """
    candidates = []

    for i in range(n_layers):
        for j in range(i + min_gap, n_layers):
            score = metric_matrix[i, j]
            candidates.append((i, j, score))

    # Sort by score (lower is better)
    candidates.sort(key=lambda x: x[2])

    return candidates[:top_k]


print("\n" + "=" * 80)
print("BEST SKIP CANDIDATES BY METRIC")
print("=" * 80)
print("\n** Notation: Layer X → Y means skipping from layer X to layer Y **")
print("   (skips all layers from X+1 through Y inclusive)")
print("   Example: Layer 9 → 11 means we skip layers 10 AND 11 = 2 layers skipped")
print("=" * 80)

for metric_key in skip_metrics_np.keys():
    print(f"\n{metric_key.upper().replace('_', ' ')}:")
    print("-" * 60)

    best_skips = find_best_skips(skip_metrics_np[metric_key], original_num_layers, top_k=5, min_gap=1)

    for rank, (src, tgt, score) in enumerate(best_skips, 1):
        layers_skipped = tgt - src  # Total number of layers we're skipping
        print(
            f"  {rank}. Layer {src:2d} → {tgt:2d}  [skips {layers_skipped} layer{'s' if layers_skipped > 1 else ''}]  Score: {score:.4f}"
        )

In [ ]:
# Compute correlation matrix between metrics
from scipy.stats import spearmanr

# Flatten all metric matrices (only upper triangle to avoid diagonal)
n = original_num_layers
triu_indices = np.triu_indices(n, k=1)

metric_names = list(skip_metrics_np.keys())
flattened_metrics = {}

for name in metric_names:
    matrix = skip_metrics_np[name]
    flattened_metrics[name] = matrix[triu_indices]

# Compute pairwise correlations
n_metrics = len(metric_names)
correlation_matrix = np.zeros((n_metrics, n_metrics))

for i, name1 in enumerate(metric_names):
    for j, name2 in enumerate(metric_names):
        if i == j:
            correlation_matrix[i, j] = 1.0
        else:
            corr, _ = spearmanr(flattened_metrics[name1], flattened_metrics[name2])
            correlation_matrix[i, j] = corr

# Plot correlation heatmap
fig, ax = plt.subplots(figsize=(10, 8))

sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".3f",
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    xticklabels=[name.replace("_", "\n") for name in metric_names],
    yticklabels=[name.replace("_", "\n") for name in metric_names],
    ax=ax,
    cbar_kws={"label": "Spearman Correlation"},
)

ax.set_title(
    f"Correlation Between Skip Quality Metrics\n{model_name} on {dataset_name}", fontsize=14, fontweight="bold", pad=20
)

plt.tight_layout()
plt.show()

# Save
save_path = PLOTS_DIR / f"{dataset_name}_{model_name.split('/')[1]}_metric_correlations.png"
fig.savefig(save_path, dpi=300, bbox_inches="tight")
print(f"\nSaved to: {save_path}")

In [ ]:
import pandas as pd

print("\n" + "=" * 80)
print("METRIC SUMMARY STATISTICS")
print("=" * 80)

summary_data = []

for name, matrix in skip_metrics_np.items():
    # Get upper triangle (exclude diagonal)
    triu_vals = matrix[np.triu_indices_from(matrix, k=1)]

    summary_data.append(
        {
            "Metric": name.replace("_", " ").title(),
            "Min": f"{triu_vals.min():.4f}",
            "Max": f"{triu_vals.max():.4f}",
            "Mean": f"{triu_vals.mean():.4f}",
            "Std": f"{triu_vals.std():.4f}",
            "Median": f"{np.median(triu_vals):.4f}",
        }
    )

summary_df = pd.DataFrame(summary_data)
print("\n", summary_df.to_string(index=False))

print("\n" + "=" * 80)

In [ ]:
import time


def compute_linear_error_all_pairs(layers):
    """
    Compute Linear Approximation Error for ALL layer pairs (including i >= j for completeness).

    Args:
        layers: [n_layers, n_samples, n_tokens, n_features] tensor

    Returns:
        error_matrix: [n_layers, n_layers] with errors for all pairs
        skip_info: List of dicts with detailed information about each skip
    """
    n_layers = layers.shape[0]
    error_matrix = torch.zeros((n_layers, n_layers))
    skip_info = []

    # Total pairs (including diagonal and both triangles)
    total_pairs = n_layers * n_layers
    computed = 0

    print(f"Computing Linear Approximation Error for ALL {total_pairs} layer pairs...")
    print(f"This includes:")
    print(f"  - Diagonal (i=j): {n_layers} pairs (identity mappings)")
    print(f"  - Upper triangle (i<j): {n_layers * (n_layers - 1) // 2} pairs (forward skips)")
    print(f"  - Lower triangle (i>j): {n_layers * (n_layers - 1) // 2} pairs (backward mappings)")
    print()

    start_time = time.time()

    for i in range(n_layers):
        for j in range(n_layers):
            layer_i = layers[i].to(device)
            layer_j = layers[j].to(device)

            # Compute error
            error = linear_approximation_error(layer_i, layer_j)
            error_matrix[i, j] = error

            # Store detailed info (only for forward skips i < j)
            if i < j:
                skip_info.append(
                    {
                        "source": i,
                        "target": j,
                        "layers_skipped": j - i,
                        "error": error,
                    }
                )

            computed += 1

            # Progress reporting
            if computed % 20 == 0 or computed == total_pairs:
                elapsed = time.time() - start_time
                print(
                    f"  Progress: {computed}/{total_pairs} pairs ({100*computed/total_pairs:.1f}%) - {elapsed:.1f}s elapsed"
                )

    total_time = time.time() - start_time
    print(f"\nDone! Total time: {total_time:.2f}s ({total_time/total_pairs*1000:.1f}ms per pair)")

    return error_matrix, skip_info


# Compute linear error for all pairs
linear_error_matrix, skip_details = compute_linear_error_all_pairs(layer_embeddings_stacked)
linear_error_np = linear_error_matrix.cpu().numpy()

print(f"\nError matrix shape: {linear_error_np.shape}")
print(f"Error range: [{linear_error_np.min():.4f}, {linear_error_np.max():.4f}]")

In [ ]:
# Analyze error by skip distance
skip_distance_data = {i: [] for i in range(1, original_num_layers)}

for info in skip_details:
    distance = info["layers_skipped"]
    error = info["error"]
    skip_distance_data[distance].append(error)

# Create box plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Box plot
distances = []
errors_by_distance = []
for distance in sorted(skip_distance_data.keys()):
    if skip_distance_data[distance]:
        distances.append(distance)
        errors_by_distance.append(skip_distance_data[distance])

bp = ax1.boxplot(
    errors_by_distance,
    positions=distances,
    widths=0.6,
    patch_artist=True,
    boxprops=dict(facecolor="lightblue", alpha=0.7),
    medianprops=dict(color="red", linewidth=2),
    whiskerprops=dict(linewidth=1.5),
    capprops=dict(linewidth=1.5),
)

ax1.set_xlabel("Number of Layers Skipped", fontsize=12, fontweight="bold")
ax1.set_ylabel("Linear Approximation Error", fontsize=12, fontweight="bold")
ax1.set_title("Error Distribution by Skip Distance", fontsize=14, fontweight="bold")
ax1.grid(True, alpha=0.3)
ax1.set_xticks(distances)

# Plot 2: Mean error with error bars (std)
mean_errors = [np.mean(skip_distance_data[d]) for d in distances]
std_errors = [np.std(skip_distance_data[d]) for d in distances]

ax2.errorbar(
    distances,
    mean_errors,
    yerr=std_errors,
    marker="o",
    markersize=8,
    capsize=5,
    capthick=2,
    linewidth=2,
    color="darkblue",
    ecolor="lightblue",
    label="Mean ± Std",
)
ax2.fill_between(
    distances,
    [m - s for m, s in zip(mean_errors, std_errors)],
    [m + s for m, s in zip(mean_errors, std_errors)],
    alpha=0.2,
    color="blue",
)

ax2.set_xlabel("Number of Layers Skipped", fontsize=12, fontweight="bold")
ax2.set_ylabel("Linear Approximation Error", fontsize=12, fontweight="bold")
ax2.set_title("Mean Error by Skip Distance", fontsize=14, fontweight="bold")
ax2.grid(True, alpha=0.3)
ax2.set_xticks(distances)
ax2.legend(fontsize=11)

plt.suptitle(
    f"Linear Approximation Error Analysis\n{model_name} on {dataset_name}", fontsize=15, fontweight="bold", y=1.02
)
plt.tight_layout()
plt.show()

# Save
save_path = PLOTS_DIR / f"{dataset_name}_{model_name.split('/')[1]}_error_by_distance.png"
fig.savefig(save_path, dpi=300, bbox_inches="tight")
print(f"Saved to: {save_path}")

In [ ]:
# Create comprehensive ranking table
skip_df = pd.DataFrame(skip_details)
skip_df = skip_df.sort_values("error")
skip_df["rank"] = range(1, len(skip_df) + 1)

# Display top 20 candidates
print("=" * 80)
print("TOP 20 SKIP CANDIDATES (Lowest Linear Approximation Error)")
print("=" * 80)
print(f"\nModel: {model_name}")
print(f"Dataset: {dataset_name}")
print(f"Samples: {max_samples}\n")

top_20 = skip_df.head(200).copy()
top_20["skip_notation"] = top_20.apply(lambda row: f"{int(row['source']):2d} → {int(row['target']):2d}", axis=1)

display_df = top_20[["rank", "skip_notation", "layers_skipped", "error"]].copy()
display_df.columns = ["Rank", "Skip (Src → Tgt)", "Layers Skipped", "Error"]

print(display_df.to_string(index=False))
print("\n" + "=" * 80)

# Also show statistics by skip distance
print("\nSTATISTICS BY SKIP DISTANCE:")
print("-" * 60)
for distance in sorted(skip_distance_data.keys()):
    if skip_distance_data[distance]:
        errors = skip_distance_data[distance]
        print(
            f"  {distance} layer{'s' if distance > 1 else ' '} skipped: "
            f"mean={np.mean(errors):.4f}, "
            f"std={np.std(errors):.4f}, "
            f"min={np.min(errors):.4f}, "
            f"max={np.max(errors):.4f}, "
            f"n={len(errors)} pairs"
        )

In [ ]:
# For each source layer, what's the best target?
best_targets = []
best_errors = []
for i in range(original_num_layers - 1):
    # Find best target layer for this source
    possible_targets = [(j, linear_error_np[i, j]) for j in range(i + 1, original_num_layers)]
    if possible_targets:
        best_j, best_error = min(possible_targets, key=lambda x: x[1])
        best_targets.append(best_j)
        best_errors.append(best_error)
    else:
        best_targets.append(None)
        best_errors.append(None)

# Create plot
fig, ax = plt.subplots(figsize=(14, 6))

source_layers = list(range(original_num_layers - 1))
colors = plt.cm.RdYlGn_r([e if e is not None else 0 for e in best_errors])

bars = ax.bar(source_layers, best_errors, color=colors, edgecolor="black", linewidth=1.5, alpha=0.8)

# Add text labels showing best target
for i, (target, error) in enumerate(zip(best_targets, best_errors)):
    if target is not None:
        layers_skipped = target - i
        ax.text(
            i, error + 0.02, f"→{target}\n({layers_skipped}L)", ha="center", va="bottom", fontsize=9, fontweight="bold"
        )

ax.set_xlabel("Source Layer", fontsize=12, fontweight="bold")
ax.set_ylabel("Best Skip Error", fontsize=12, fontweight="bold")
ax.set_title(
    f"Best Skip Target for Each Layer (Lowest Error)\n{model_name} on {dataset_name}", fontsize=14, fontweight="bold"
)
ax.set_xticks(source_layers)
ax.grid(True, alpha=0.3, axis="y")
ax.set_ylim(0, max(best_errors) * 1.2)

plt.tight_layout()
plt.show()

# Save
save_path = PLOTS_DIR / f"{dataset_name}_{model_name.split('/')[1]}_best_skip_per_layer.png"
fig.savefig(save_path, dpi=300, bbox_inches="tight")
print(f"Saved to: {save_path}")

In [ ]:
# For each layer, show error to all possible targets
fig, ax = plt.subplots(figsize=(14, 8))

for i in range(original_num_layers - 1):
    targets = list(range(i + 1, original_num_layers))
    errors = [linear_error_np[i, j] for j in targets]

    # Plot line for this source layer
    ax.plot(targets, errors, marker="o", linewidth=2, markersize=6, label=f"Source layer: {i}", alpha=0.7)

# Add threshold line for acceptable skip quality
# threshold = 0.3
# ax.axhline(y=threshold, color='red', linestyle='--', linewidth=2.5,
#            label=f'Quality Threshold (ε={threshold})', alpha=0.8, zorder=10)

ax.set_xlabel("Target Layer", fontsize=12, fontweight="bold")
ax.set_ylabel("Linear Approximation Error", fontsize=12, fontweight="bold")
ax.set_title(
    f"Error from Each Source Layer to All Target Layers\n{model_name} on {dataset_name}", fontsize=14, fontweight="bold"
)
ax.grid(True, alpha=0.3)

# Set x-ticks to show all layers
ax.set_xticks(range(original_num_layers))

# Legend in 1 column on the right
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=9, ncol=1)

plt.tight_layout()
plt.show()

# Save
save_path = PLOTS_DIR / f"{dataset_name}_{model_name.split('/')[1]}_error_all_targets.png"
fig.savefig(save_path, dpi=300, bbox_inches="tight")
print(f"Saved to: {save_path}")